# EuroSAT RGB vs multispectral — Colab driver

Runs the full experiment matrix on a free **T4 GPU** (~5-7 GPU-hours). Idempotent and resumable: if the session drops, just re-run from the top — completed runs skip themselves.

**Before running:** Runtime -> Change runtime type -> GPU (T4). Then Runtime -> Run all.

Set `GITHUB_REPO` below to your repository URL (and add a token for a private repo).

In [ ]:
#@title 1. Clone the code repository
GITHUB_REPO = "https://github.com/Mohammed-AlKuhali/eurosat-ms-classification.git"  #@param {type:"string"}
import os
if not os.path.isdir('eurosat-ms-classification'):
    !git clone $GITHUB_REPO
%cd eurosat-ms-classification
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
#@title 2. Install dependencies
!pip install -q -e . >/dev/null
!pip install -q torch torchvision timm==1.0.11 torchgeo==0.6.2 grad-cam==1.5.4 statsmodels scikit-image >/dev/null
import torch; print('CUDA available:', torch.cuda.is_available())

In [ ]:
#@title 3. Prepare the dataset (from your uploaded zip — no Drive needed)
import os, glob
LOCAL_ZIP = '/content/EuroSAT_MS.zip'  #@param {type:"string"}
os.makedirs('data/raw', exist_ok=True)
if not glob.glob('data/raw/EuroSAT_MS/*/*.tif'):
    if os.path.exists(LOCAL_ZIP):
        print('Extracting uploaded zip (no Drive needed)...')
        !unzip -q -o "$LOCAL_ZIP" -d data/raw
    else:
        print('No local zip found at', LOCAL_ZIP, '- downloading from Zenodo...')
        !python scripts/download_data.py --root data/raw
# Resolve the data root (zip extracts to data/raw/EuroSAT_MS/<class>/*.tif)
root = 'data/raw/EuroSAT_MS'
if not glob.glob(root + '/*/*.tif'):
    hits = glob.glob('data/raw/**/AnnualCrop', recursive=True)
    root = os.path.dirname(hits[0]) if hits else root
os.environ['EUROSAT_DATA_ROOT'] = root
n = len(glob.glob(root + '/*/*.tif'))
print(f'EUROSAT_DATA_ROOT={root}  ({n} patches)')
assert n == 27000, f'expected 27000 patches, found {n} - check the zip'

In [ ]:
#@title 4. Results stay local (Drive is full)
# Results are written to ./results in this session. Download them with the last
# cell before the session ends (or periodically). No Drive used.
!mkdir -p results/metrics results/predictions results/figures results/tables checkpoints
print('Results -> ./results  (download via the final cell before disconnecting)')

In [ ]:
#@title 5. Run the full matrix (priority order; resumable)
import os
os.environ['DEVICE'] = 'cuda'
!bash scripts/run_all.sh

In [ ]:
## Important: no Drive this session

Drive was full, so results live only in this session's `./results` folder. Two consequences:

1. **Download before you disconnect.** Run the **Download results** cell (cell 6) when the matrix finishes — or periodically as a backup. Send me the `results_bundle.zip` (or unzip it into the repo's `results/`) and I'll ingest the numbers.
2. **If the session drops mid-run, local results are lost** (nothing to resume from). Mitigation is built in: `run_all.sh` runs in **priority order**, so the mandatory comparison (E1 RGB / E2 multispectral / E3 indices) completes first. If you have to stop early, run the download cell — whatever finished is already valuable.

To resume in a fresh session: re-upload `EuroSAT_MS.zip`, re-run cells 1-5. Completed runs in the current `./results` skip themselves; lost ones re-run.

## If a session times out
Re-run cells 1-5. Completed arm/seed/fraction runs are detected from `results/metrics/*.json` and skipped, so each new session only does the remaining work. With results symlinked to Drive (cell 4), nothing is lost.

## After all runs complete
Commit the `results/` folder back to the repo (metrics + prediction CSVs are small), then run the analysis notebook locally:
```bash
git add results && git commit -m 'Add Colab experiment results'
```